# Sales Forecasting Across Multiple Retail Stores — Rossmann Pharmaceuticals
### Internship Project — NextHikes IT Solutions

**Prepared by:** Rehana

## Project Objective
Rossmann Pharmaceuticals operates stores in several cities and currently relies on
individual store managers' experience and judgement to forecast sales. The finance team
wants a **data-driven, end-to-end sales forecasting system** that predicts daily sales up to
**six weeks ahead**, taking into account promotions, competition, holidays, seasonality and
store locality.

This notebook covers:
1. Data understanding & cleaning
2. Exploratory Data Analysis (customer purchasing behaviour)
3. Feature engineering
4. Machine Learning modelling (Random Forest regression pipeline)
5. Deep Learning modelling (LSTM time-series model)
6. Model serialization & experiment tracking (MLflow)
7. Key findings, business insights and recommendations


## 1. Import Libraries

In [ ]:

import warnings, logging, json, os, pickle
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

# ---- logging setup (per project requirement 1.2) ----
logging.basicConfig(
    filename="rossmann_pipeline.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)
logger = logging.getLogger("rossmann")
logger.info("Notebook run started.")
print("Libraries loaded.")


: 

## 2. Dataset Description

| File | Rows | Description |
|---|---|---|
| `train.csv` | ~1.017M | Daily sales history per store (2013–2015) with Sales & Customers |
| `test.csv` | ~41K | Same features minus Sales/Customers — sales to be predicted 6 weeks ahead |
| `store.csv` | 1,115 | Static metadata per store: type, assortment, competition, promo2 |


In [ ]:

train = pd.read_csv("train.csv", low_memory=False, parse_dates=["Date"])
test = pd.read_csv("test.csv", low_memory=False, parse_dates=["Date"])
store = pd.read_csv("store.csv", low_memory=False)

logger.info(f"Loaded train {train.shape}, test {test.shape}, store {store.shape}")
print("train:", train.shape, "| test:", test.shape, "| store:", store.shape)


## 3. Data Understanding / Exploration

In [ ]:

train.info()


In [ ]:

train.describe(include="all").T


In [ ]:

print("Missing values - train:")
print(train.isnull().sum())
print("\nMissing values - test:")
print(test.isnull().sum())
print("\nMissing values - store:")
print(store.isnull().sum())


In [ ]:

print("Duplicate rows - train:", train.duplicated().sum())
print("Duplicate rows - test:", test.duplicated().sum())
print("Duplicate rows - store:", store.duplicated(subset=['Store']).sum())


In [ ]:

print("StateHoliday unique values:", train.StateHoliday.unique())
print("StoreType unique values:", store.StoreType.unique())
print("Assortment unique values:", store.Assortment.unique())
print("\nRows where store is closed (Open=0):", (train.Open == 0).sum())
print("Of those, rows with non-zero Sales:", ((train.Open == 0) & (train.Sales != 0)).sum())
print("\nRows where store is open but Sales == 0:", ((train.Open == 1) & (train.Sales == 0)).sum())


**Observations:**
- No duplicate rows in any of the three files.
- `StateHoliday` is stored inconsistently as the string `"0"` mixed with numeric-like values — needs
  to be treated as a categorical string.
- Closed stores (`Open == 0`) always show `Sales == 0`, which is expected and consistent —
  these rows carry no forecasting signal and will be excluded before modelling.
- `store.csv` has missing values in `CompetitionDistance` (3), `CompetitionOpenSince*` (354),
  and `Promo2Since*`/`PromoInterval` (544, i.e. stores not enrolled in Promo2).
- `test.csv` has 11 missing values in `Open` — these need to be imputed (most likely open, since
  they are weekday rows for stores that are normally open).


## 4. Data Cleaning and Preprocessing

In [ ]:

# --- Merge store metadata into train/test ---
train_m = train.merge(store, on="Store", how="left")
test_m = test.merge(store, on="Store", how="left")
print("Merged train:", train_m.shape, "| Merged test:", test_m.shape)


In [ ]:

# --- Handle missing 'Open' in test: assume open unless it's a known-closed pattern ---
# Rationale: all 11 missing rows belong to store 622 on weekdays around a known operating period,
# so the safest, most defensible assumption is that the store was open (matches its typical weekday pattern).
test_m["Open"] = test_m["Open"].fillna(1)

# --- CompetitionDistance: 3 missing values. Impute with the median (robust to outliers) ---
median_dist = train_m["CompetitionDistance"].median()
train_m["CompetitionDistance"] = train_m["CompetitionDistance"].fillna(median_dist)
test_m["CompetitionDistance"] = test_m["CompetitionDistance"].fillna(median_dist)

# --- CompetitionOpenSinceMonth/Year: missing = no known competitor yet -> fill with 0 (flag), not mean ---
for col in ["CompetitionOpenSinceMonth", "CompetitionOpenSinceYear"]:
    train_m[col] = train_m[col].fillna(0)
    test_m[col] = test_m[col].fillna(0)

# --- Promo2Since*/PromoInterval: missing = store not enrolled in Promo2 (Promo2==0), fill accordingly ---
for col in ["Promo2SinceWeek", "Promo2SinceYear"]:
    train_m[col] = train_m[col].fillna(0)
    test_m[col] = test_m[col].fillna(0)
train_m["PromoInterval"] = train_m["PromoInterval"].fillna("None")
test_m["PromoInterval"] = test_m["PromoInterval"].fillna("None")

logger.info("Missing value imputation completed.")
print("Remaining nulls in train_m:\n", train_m.isnull().sum().sum())
print("Remaining nulls in test_m:\n", test_m.isnull().sum().sum())


In [ ]:

# --- Remove rows where the store is closed: no forecasting signal, would bias averages ---
before = train_m.shape[0]
train_clean = train_m[train_m.Open == 1].copy()
after = train_clean.shape[0]
print(f"Removed {before - after:,} closed-store rows from training data ({(before-after)/before:.1%}).")
logger.info(f"Dropped {before-after} closed-store rows.")


In [ ]:

# --- Outlier check on Sales (for open stores only) ---
q1, q3 = train_clean.Sales.quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 3*iqr
outliers = (train_clean.Sales > upper).sum()
print(f"Sales upper outlier bound (3xIQR): {upper:,.0f} | rows flagged as extreme: {outliers:,} ({outliers/len(train_clean):.2%})")
# Decision: keep these — they represent genuine high-traffic days (promos/holidays), not data errors.


**Cleaning decisions & justification:**
- Closed-store rows dropped from training (constant zero target, no signal, would distort averages/models).
- `CompetitionDistance` nulls (3 rows) imputed with median — too few rows to justify dropping, and median is
  robust to the right-skew in this feature.
- `CompetitionOpenSince*` and `Promo2Since*` nulls filled with 0 rather than mean, because the null pattern
  itself is informative (no known competitor / not enrolled in Promo2) — replacing with the true mean would
  fabricate a fictitious competitor-opening date.
- Extreme high-Sales days retained: manual inspection shows they coincide with promotions/holidays, i.e.
  genuine business events, not data entry errors.


## 5. Exploratory Data Analysis — Customer Purchasing Behaviour
Each sub-section below directly answers one of the guiding questions from the project brief.

### 5.1 Are promotions distributed similarly in train vs. test?

In [ ]:

promo_train = train_m["Promo"].value_counts(normalize=True).sort_index()
promo_test = test_m["Promo"].value_counts(normalize=True).sort_index()

comp = pd.DataFrame({"Train": promo_train, "Test": promo_test})
print(comp)

comp.plot(kind="bar", figsize=(6,4))
plt.title("Promo Distribution: Train vs Test")
plt.xlabel("Promo (0=No, 1=Yes)")
plt.ylabel("Proportion of days")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("plots/promo_train_test.png", dpi=120)
plt.show()


**Insight:** The proportion of promo vs non-promo days is very close between train and test (within a couple of percentage points), so the test period is not systematically over/under-promoted relative to training history — models trained on train should generalize to test's promo mix.

### 5.2 Sales behaviour before, during and after holidays

In [ ]:

df = train_clean.sort_values(["Store", "Date"]).copy()
df["StateHoliday"] = df["StateHoliday"].astype(str)
df["is_holiday"] = df["StateHoliday"] != "0"

# Flag the day before / after a holiday, per store
df["next_day_holiday"] = df.groupby("Store")["is_holiday"].shift(-1).fillna(False)
df["prev_day_holiday"] = df.groupby("Store")["is_holiday"].shift(1).fillna(False)

def period(row):
    if row["is_holiday"]:
        return "During Holiday"
    elif row["next_day_holiday"]:
        return "Day Before Holiday"
    elif row["prev_day_holiday"]:
        return "Day After Holiday"
    else:
        return "Regular Day"

df["holiday_period"] = df.apply(period, axis=1)
summary = df.groupby("holiday_period")["Sales"].agg(["mean", "median", "count"]).sort_values("mean", ascending=False)
print(summary)

plt.figure(figsize=(8,5))
sns.barplot(x=summary.index, y=summary["mean"])
plt.title("Average Sales: Before / During / After Holidays vs Regular Days")
plt.ylabel("Average Sales")
plt.xlabel("")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("plots/holiday_sales.png", dpi=120)
plt.show()


**Insight:** Sales spike sharply on the day *before* a holiday (stock-up behaviour) and drop during the holiday itself (most stores are closed on state holidays, so the few open ones see reduced footfall). Sales normalize quickly in the days after.

### 5.3 Seasonal purchase behaviour (Christmas, Easter, etc.)

In [ ]:

df["Month"] = df["Date"].dt.month
df["Year"] = df["Date"].dt.year

monthly = df.groupby(["Year","Month"])["Sales"].mean().reset_index()
monthly["Period"] = monthly["Year"].astype(str) + "-" + monthly["Month"].astype(str).str.zfill(2)

plt.figure(figsize=(14,5))
plt.plot(monthly["Period"], monthly["Sales"], marker="o")
plt.xticks(rotation=90)
plt.title("Average Daily Sales by Month (2013-2015)")
plt.ylabel("Average Sales")
plt.tight_layout()
plt.savefig("plots/monthly_seasonality.png", dpi=120)
plt.show()


In [ ]:

# Zoom on StateHoliday types: a=public holiday, b=Easter, c=Christmas
holiday_type_sales = df[df.is_holiday].groupby("StateHoliday")["Sales"].mean()
print("Average sales by holiday type (a=public, b=Easter, c=Christmas):")
print(holiday_type_sales)


**Insight:** Clear year-end (December) peaks are visible each year, consistent with Christmas shopping, with a smaller lift around Easter. This confirms strong seasonality that a forecasting model needs to capture explicitly (e.g. via month/holiday-proximity features), not just recent trend.

### 5.4 Correlation between Sales and Customers

In [ ]:

corr = train_clean[["Sales","Customers"]].corr().iloc[0,1]
print(f"Correlation (Sales, Customers): {corr:.3f}")

plt.figure(figsize=(6,5))
sample = train_clean.sample(20000, random_state=42)
sns.scatterplot(data=sample, x="Customers", y="Sales", alpha=0.15, s=10)
plt.title(f"Sales vs Customers (r = {corr:.2f})")
plt.tight_layout()
plt.savefig("plots/sales_vs_customers.png", dpi=120)
plt.show()


**Insight:** Sales and Customers are strongly positively correlated (r ≈ 0.8+), which makes sense — more footfall drives more revenue. However, the relationship isn't perfectly linear: at a given customer count, Sales varies (spending-per-customer differs by promo activity, store type, day of week), so Customers alone won't fully explain Sales — average basket size matters too.

### 5.5 How does Promo affect Sales and Customers?

In [ ]:

promo_effect = train_clean.groupby("Promo")[["Sales","Customers"]].mean()
promo_effect["Sales_per_Customer"] = promo_effect["Sales"] / promo_effect["Customers"]
print(promo_effect)

fig, axes = plt.subplots(1,3, figsize=(15,4))
train_clean.groupby("Promo")["Sales"].mean().plot(kind="bar", ax=axes[0], title="Avg Sales")
train_clean.groupby("Promo")["Customers"].mean().plot(kind="bar", ax=axes[1], title="Avg Customers")
promo_effect["Sales_per_Customer"].plot(kind="bar", ax=axes[2], title="Avg Sales per Customer")
for ax in axes: ax.set_xlabel("Promo (0=No,1=Yes)"); ax.set_xticklabels(["No","Yes"], rotation=0)
plt.tight_layout()
plt.savefig("plots/promo_effect.png", dpi=120)
plt.show()


**Insight:** Promo days show higher average Sales *and* higher average Customers *and* a higher Sales-per-Customer ratio. This means promos both attract more footfall **and** increase basket size of existing customers — the effect isn't just more people buying the same amount, it's a genuine demand lift on both fronts.

### 5.6 Which stores benefit most from promos? (targeting opportunity)

In [ ]:

store_promo_lift = train_clean.groupby(["Store","Promo"])["Sales"].mean().unstack()
store_promo_lift["lift_pct"] = (store_promo_lift[1] - store_promo_lift[0]) / store_promo_lift[0] * 100
store_promo_lift = store_promo_lift.dropna().sort_values("lift_pct", ascending=False)
print("Top 10 stores by promo sales lift (%):")
print(store_promo_lift.head(10))
print("\nBottom 10 stores by promo sales lift (%):")
print(store_promo_lift.tail(10))

plt.figure(figsize=(10,4))
sns.histplot(store_promo_lift["lift_pct"], bins=40, kde=True)
plt.axvline(store_promo_lift["lift_pct"].median(), color="red", linestyle="--", label="Median lift")
plt.title("Distribution of Per-Store Promo Sales Lift (%)")
plt.xlabel("% Sales lift on promo days vs non-promo days")
plt.legend()
plt.tight_layout()
plt.savefig("plots/store_promo_lift.png", dpi=120)
plt.show()


**Insight:** Promo lift varies widely by store — some stores see 40%+ lift while others see little to no benefit. This is a direct, actionable finding: promo budget would be more efficiently spent by **targeting high-lift stores** rather than running promos uniformly across all stores.

### 5.7 Customer trends around store open/closing

In [ ]:

dow_sales = train_clean.groupby("DayOfWeek")[["Sales","Customers"]].mean()
dow_sales.index = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
print(dow_sales)

dow_sales.plot(kind="bar", secondary_y="Customers", figsize=(9,5))
plt.title("Average Sales & Customers by Day of Week")
plt.tight_layout()
plt.savefig("plots/dow_sales.png", dpi=120)
plt.show()


**Insight:** Sales and customer counts are highest on Mondays (start-of-week stock-up) and, where stores are open, on weekends. Sunday openings are rare — most stores are closed — but the few that open show strong Sunday footfall.

### 5.8 Stores open on all weekdays — does that affect weekend sales?

In [ ]:

weekday_open_counts = train.groupby("Store").apply(
    lambda g: g[g.DayOfWeek.isin([1,2,3,4,5])]["Open"].mean()
)
always_open_weekday_stores = weekday_open_counts[weekday_open_counts == 1.0].index
print(f"{len(always_open_weekday_stores)} stores are open on every weekday in the dataset.")

sunday_data = train_clean[train_clean.DayOfWeek == 7]
sunday_open_stores = sunday_data["Store"].unique()
print(f"{len(sunday_open_stores)} stores open at least once on a Sunday.")

overlap = set(always_open_weekday_stores) & set(sunday_open_stores)
print(f"Of the always-open-on-weekdays stores, {len(overlap)} also open on Sundays.")

comparison = train_clean[train_clean.Store.isin(always_open_weekday_stores)].groupby("DayOfWeek")["Sales"].mean()
comparison.index = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"][:len(comparison)]
print(comparison)


**Insight:** Stores that never close on weekdays are a small subset, and only a fraction of them also open on Sundays. Where they do open on Sunday, average sales are competitive with weekday levels — suggesting Sunday opening is a viable, underused revenue opportunity for stores that currently stay closed.

### 5.9 How does Assortment type affect Sales?

In [ ]:

assort_sales = train_clean  # Assortment already present from the store merge
assort_summary = assort_sales.groupby("Assortment")["Sales"].agg(["mean","median","count"])
print(assort_summary)

plt.figure(figsize=(7,5))
sns.boxplot(data=assort_sales, x="Assortment", y="Sales", showfliers=False)
plt.title("Sales Distribution by Assortment Type (a=basic, b=extra, c=extended)")
plt.tight_layout()
plt.savefig("plots/assortment_sales.png", dpi=120)
plt.show()


**Insight:** 'Extra' assortment (b) stores show noticeably higher average sales than basic (a) or extended (c), though they are also far fewer in number — this could reflect either the assortment itself or that these are larger-format stores; it's a correlation worth flagging to the business rather than a proven causal driver.

### 5.10 Does distance to the nearest competitor affect Sales?

In [ ]:

comp_sales = train_clean.dropna(subset=["CompetitionDistance"])
comp_sales["dist_bucket"] = pd.qcut(comp_sales["CompetitionDistance"], 5,
                                     labels=["Very Close","Close","Medium","Far","Very Far"])
bucket_summary = comp_sales.groupby("dist_bucket")["Sales"].mean()
print(bucket_summary)

corr_dist = comp_sales[["CompetitionDistance","Sales"]].corr().iloc[0,1]
print(f"\nCorrelation (CompetitionDistance, Sales): {corr_dist:.3f}")

plt.figure(figsize=(8,5))
bucket_summary.plot(kind="bar")
plt.title("Average Sales by Competitor Distance Bucket")
plt.ylabel("Average Sales")
plt.tight_layout()
plt.savefig("plots/competition_distance.png", dpi=120)
plt.show()


**Insight:** Counter-intuitively, stores with *closer* competitors tend to show slightly **higher** average sales, and the raw correlation is weak. This is a well-known pattern in retail: competitors cluster in high-footfall areas (city centres), so proximity is confounded with location quality. Distance alone is not a reliable standalone predictor — it needs to be considered alongside store location/type rather than in isolation.

### 5.11 Effect of a new competitor opening near an existing store

In [ ]:

comp_open = store.dropna(subset=["CompetitionOpenSinceYear","CompetitionOpenSinceMonth"]).copy()
comp_open["CompetitionOpenDate"] = pd.to_datetime(
    comp_open["CompetitionOpenSinceYear"].astype(int).astype(str) + "-" +
    comp_open["CompetitionOpenSinceMonth"].astype(int).astype(str) + "-01",
    errors="coerce"
)

# pick a few stores whose competitor opened *during* the train date range, so before/after is observable
mask = comp_open["CompetitionOpenDate"].between(train.Date.min(), train.Date.max())
candidate_stores = comp_open[mask]["Store"].tolist()
print(f"{len(candidate_stores)} stores had a new competitor open during the observed period.")

example_results = []
for s in candidate_stores[:200]:
    open_date = comp_open.loc[comp_open.Store == s, "CompetitionOpenDate"].values[0]
    sdata = train_clean[train_clean.Store == s].copy()
    before = sdata[sdata.Date < open_date]["Sales"].mean()
    after = sdata[sdata.Date >= open_date]["Sales"].mean()
    if pd.notnull(before) and pd.notnull(after) and before > 0:
        example_results.append({"Store": s, "before": before, "after": after,
                                 "pct_change": (after-before)/before*100})

res_df = pd.DataFrame(example_results).dropna()
print(f"Average sales change after new competitor opens (n={len(res_df)} stores): "
      f"{res_df['pct_change'].mean():.1f}%")
print(res_df["pct_change"].describe())


**Insight:** On average, sales do **not** collapse after a nearby competitor opens — the mean change is close to flat/slightly negative across the sampled stores, with wide variation. This suggests store loyalty and other factors (location, assortment) buffer most stores from new competition in the short term, though a subset of stores clearly are more exposed.

## 6. Feature Engineering (Task 2.1 — Preprocessing)

In [ ]:

def engineer_features(df, holiday_dates):
    df = df.copy()
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["Day"] = df["Date"].dt.day
    df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
    df["IsWeekend"] = (df["DayOfWeek"] >= 6).astype(int)

    # month position
    df["MonthPeriod"] = np.select(
        [df["Day"] <= 10, df["Day"] <= 20],
        ["Beginning", "Mid"],
        default="End"
    )

    # days to / after nearest holiday
    holiday_dates_sorted = np.sort(holiday_dates.values)
    def days_to_next_holiday(d):
        future = holiday_dates_sorted[holiday_dates_sorted >= np.datetime64(d)]
        return (future.min() - np.datetime64(d)).astype('timedelta64[D]').astype(int) if len(future) else 999
    def days_after_last_holiday(d):
        past = holiday_dates_sorted[holiday_dates_sorted <= np.datetime64(d)]
        return (np.datetime64(d) - past.max()).astype('timedelta64[D]').astype(int) if len(past) else 999

    unique_dates = df["Date"].unique()
    to_next = {d: days_to_next_holiday(d) for d in unique_dates}
    after_last = {d: days_after_last_holiday(d) for d in unique_dates}
    df["DaysToHoliday"] = df["Date"].map(to_next).clip(upper=60)
    df["DaysAfterHoliday"] = df["Date"].map(after_last).clip(upper=60)

    # competition open duration in months (0 if unknown/not yet open)
    comp_since = pd.to_datetime(
        dict(year=df["CompetitionOpenSinceYear"].replace(0, np.nan),
             month=df["CompetitionOpenSinceMonth"].replace(0, np.nan), day=1),
        errors="coerce"
    )
    months_active = ((df["Date"].dt.year - comp_since.dt.year) * 12 +
                      (df["Date"].dt.month - comp_since.dt.month))
    df["CompetitionMonthsOpen"] = months_active.fillna(0).clip(lower=0)

    # Promo2 active on this date?
    promo2_active = []
    month_abbr = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}
    for _, row in df[["Promo2","Promo2SinceYear","Promo2SinceWeek","PromoInterval","Year","WeekOfYear","Month"]].iterrows():
        if row["Promo2"] == 0:
            promo2_active.append(0)
            continue
        started = (row["Year"] > row["Promo2SinceYear"]) or \
                  (row["Year"] == row["Promo2SinceYear"] and row["WeekOfYear"] >= row["Promo2SinceWeek"])
        in_interval = isinstance(row["PromoInterval"], str) and month_abbr.get(row["Month"], "") in row["PromoInterval"]
        promo2_active.append(int(started and in_interval))
    df["Promo2Active"] = promo2_active

    df["StateHoliday"] = df["StateHoliday"].astype(str)
    return df

holiday_dates = train_clean.loc[train_clean["StateHoliday"] != "0", "Date"]
train_fe = engineer_features(train_clean, holiday_dates)
print("Engineered feature columns added:")
new_cols = [c for c in train_fe.columns if c not in train_clean.columns]
print(new_cols)
train_fe[new_cols].head()


**Features created:** `Year, Month, Day, WeekOfYear, IsWeekend, MonthPeriod (Beginning/Mid/End), DaysToHoliday, DaysAfterHoliday, CompetitionMonthsOpen, Promo2Active`. These directly implement the project's requested date-derived features (weekday/weekend, days to/after holiday, month position) plus two extra engineered features — `CompetitionMonthsOpen` and `Promo2Active` — that translate the raw competition/promo2 date fields into a form the model can use directly on any given day, for extra credit as suggested in the brief.

## 7. Building the ML Model (Task 2.2 — sklearn Pipeline)

In [ ]:

feature_cols_num = ["Store","DayOfWeek","Promo","SchoolHoliday","CompetitionDistance",
                     "CompetitionMonthsOpen","Promo2","Promo2Active","Year","Month","Day",
                     "WeekOfYear","IsWeekend","DaysToHoliday","DaysAfterHoliday"]
feature_cols_cat = ["StateHoliday","StoreType","Assortment","MonthPeriod"]

X = train_fe[feature_cols_num + feature_cols_cat]
y = train_fe["Sales"]

# Time-based split (last 6 weeks held out, matching the business ask of a 6-week forecast horizon)
cutoff = train_fe["Date"].max() - pd.Timedelta(weeks=6)
train_idx = train_fe["Date"] <= cutoff
val_idx = train_fe["Date"] > cutoff

X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]
print(f"Train: {X_train.shape}, Validation (last 6 weeks): {X_val.shape}")


**Note on data leakage:** the split is chronological (last 6 weeks held out), matching the real forecast horizon the business needs — a random shuffle-split would leak future information into training since sales are time-dependent, so a time-based split is the correct choice here.

In [ ]:

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), feature_cols_num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), feature_cols_cat)
])
print("Preprocessor defined (scaling numeric features, one-hot encoding categoricals).")


## 8. Model Comparison (Task 2.2/2.3 — choosing an algorithm)

Rather than committing to a single algorithm up front, we train **three different models** on the
exact same pipeline, features, and time-based train/validation split, then compare them on the same
metrics before choosing a production model:

1. **Linear Regression** — a simple baseline. Sales is driven by strong non-linear/categorical effects
   (store identity, promos, day-of-week), so we expect this to underperform — it earns its place in the
   comparison precisely by showing *how much* the non-linear models are adding.
2. **Random Forest Regressor** — a bagged ensemble of decision trees; robust, handles non-linearity and
   feature interactions natively, and gives us feature importances and empirical prediction intervals for
   free (Task 2.4).
3. **XGBoost Regressor** — a gradient-boosted tree ensemble; typically the strongest tabular-data
   performer, included as a stronger alternative to check how much accuracy is left on the table.


In [ ]:

from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

def rmspe(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.sqrt(np.mean(((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2))

candidate_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=15, min_samples_leaf=5,
                                            n_jobs=-1, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=300, max_depth=8, learning_rate=0.1,
                             subsample=0.9, colsample_bytree=0.9, tree_method="hist",
                             n_jobs=-1, random_state=42),
}

trained_pipelines = {}
comparison_rows = []

for name, model in candidate_models.items():
    logger.info(f"Training {name}...")
    pipe = Pipeline([("preprocess", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_val)

    mae_ = mean_absolute_error(y_val, preds)
    rmse_ = np.sqrt(mean_squared_error(y_val, preds))
    rmspe_ = rmspe(y_val, preds)
    r2_ = r2_score(y_val, preds)

    trained_pipelines[name] = pipe
    comparison_rows.append({"Model": name, "MAE": mae_, "RMSE": rmse_, "RMSPE": rmspe_, "R2": r2_})
    logger.info(f"{name} -> MAE={mae_:.1f}, RMSPE={rmspe_:.4f}, R2={r2_:.4f}")
    print(f"{name:<20s} MAE={mae_:>9,.1f}  RMSE={rmse_:>9,.1f}  RMSPE={rmspe_:>7.4f} ({rmspe_:.1%})  R2={r2_:>6.4f}")

comparison_df = pd.DataFrame(comparison_rows).set_index("Model")
comparison_df


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13,4.5))
comparison_df["RMSPE"].plot(kind="bar", ax=axes[0], color=["#B33A3A","#2E6E9E","#2E8B57"])
axes[0].set_title("RMSPE by Model (lower is better)")
axes[0].set_ylabel("RMSPE")
axes[0].tick_params(axis='x', rotation=20)

comparison_df["R2"].plot(kind="bar", ax=axes[1], color=["#B33A3A","#2E6E9E","#2E8B57"])
axes[1].set_title("R² by Model (higher is better)")
axes[1].set_ylabel("R2")
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig("plots/model_comparison.png", dpi=120)
plt.show()


## 9. Loss Function / Evaluation Metric (Task 2.3)

We report MAE, RMSE and R² above, but treat **RMSPE (Root Mean Squared Percentage Error)** as the
primary loss/evaluation metric for model selection — it's the metric this exact Rossmann challenge is
historically judged on, and it suits the business need well: retail managers care about *relative*
forecast error (being off by 5% matters similarly whether a store does $2,000 or $20,000 a day), which
plain RMSE does not capture since it weights high-volume stores far more heavily in absolute terms.

In [ ]:

best_model_name = comparison_df["RMSPE"].idxmin()
print(f"Best model by RMSPE: {best_model_name}")
print(comparison_df.loc[[best_model_name]])


**Model selection decision:** the gap here is not narrow — **XGBoost** beats Random Forest by roughly
12 percentage points of RMSPE (16.3% vs 28.5%) and Random Forest beats plain Linear Regression by an even
wider margin. An accuracy gap this large is not worth sacrificing for convenience, so **XGBoost is selected
as the production model** for the remainder of this notebook. XGBoost still exposes the same
`feature_importances_` interface used below (Task 2.4), and we build a genuine prediction interval for it
via **quantile regression** (two extra lightweight XGBoost models trained to predict the 5th and 95th
percentiles directly) rather than relying on Random-Forest-specific tree variance.

In [ ]:

production_pipeline = trained_pipelines["XGBoost"]
preds_val = production_pipeline.predict(X_val)

mae = mean_absolute_error(y_val, preds_val)
rmse = np.sqrt(mean_squared_error(y_val, preds_val))
rmspe_val = rmspe(y_val, preds_val)
r2 = r2_score(y_val, preds_val)

print(f"Selected production model: XGBoost")
print(f"MAE:   {mae:,.1f}")
print(f"RMSE:  {rmse:,.1f}")
print(f"RMSPE: {rmspe_val:.4f}  ({rmspe_val:.1%})")
print(f"R2:    {r2:.4f}")


In [ ]:

plt.figure(figsize=(7,6))
sample_idx = np.random.choice(len(y_val), 3000, replace=False)
plt.scatter(y_val.values[sample_idx], preds_val[sample_idx], alpha=0.2, s=10)
lims = [0, max(y_val.max(), preds_val.max())]
plt.plot(lims, lims, 'r--', label="Perfect prediction")
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title(f"Actual vs Predicted Sales (Validation, last 6 weeks) — XGBoost\nRMSPE={rmspe_val:.1%}, R2={r2:.3f}")
plt.legend()
plt.tight_layout()
plt.savefig("plots/actual_vs_predicted.png", dpi=120)
plt.show()


## 10. Post-Prediction Analysis (Task 2.4)

### 10.1 Feature Importance

In [ ]:

ohe_cols = production_pipeline.named_steps["preprocess"].named_transformers_["cat"].get_feature_names_out(feature_cols_cat)
all_feature_names = feature_cols_num + list(ohe_cols)
importances = production_pipeline.named_steps["model"].feature_importances_

fi = pd.Series(importances, index=all_feature_names).sort_values(ascending=False).head(15)
plt.figure(figsize=(9,6))
fi.sort_values().plot(kind="barh")
plt.title("Top 15 Feature Importances — XGBoost")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("plots/feature_importance.png", dpi=120)
plt.show()
print(fi)


**Insight:** `StoreType_b` is by far the single most important feature, with `Promo`, `Assortment_a`, `Promo2` and `CompetitionDistance` next. Store-format/assortment identity dominates — this makes intuitive sense: a small number of 'type b' stores are large-format outlets with structurally higher baseline sales, so the model leans heavily on that categorical signal before finer day-to-day drivers like `DayOfWeek` or `Month`. This reinforces the EDA finding that store identity and promotions are the primary sales drivers, well ahead of the engineered holiday-proximity features (`DaysToHoliday`/`DaysAfterHoliday` don't even make the top 15).

### 10.2 Prediction Confidence Interval

Random Forest's tree-variance trick doesn't apply directly to a boosted model, so we build a genuine **quantile regression** interval instead: two extra XGBoost models are trained with a pinball (quantile) loss to predict the 5th and 95th percentile of Sales directly, giving a real empirical 90% prediction interval around the point forecast.

In [ ]:

# 90% prediction interval via quantile regression (XGBoost reg:quantileerror objective)
xgb_lower = Pipeline([("preprocess", preprocessor),
                       ("model", XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1,
                                               objective="reg:quantileerror", quantile_alpha=0.05,
                                               tree_method="hist", n_jobs=-1, random_state=42))])
xgb_upper = Pipeline([("preprocess", preprocessor),
                       ("model", XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1,
                                               objective="reg:quantileerror", quantile_alpha=0.95,
                                               tree_method="hist", n_jobs=-1, random_state=42))])
xgb_lower.fit(X_train, y_train)
xgb_upper.fit(X_train, y_train)

sample_X_val = X_val.iloc[:500]
pred_mean = production_pipeline.predict(sample_X_val)
pred_lower = xgb_lower.predict(sample_X_val)
pred_upper = xgb_upper.predict(sample_X_val)
# guard against occasional quantile crossing on a handful of rows
pred_lower, pred_upper = np.minimum(pred_lower, pred_upper), np.maximum(pred_lower, pred_upper)

ci_df = pd.DataFrame({"Predicted": pred_mean, "Lower_5pct": pred_lower, "Upper_95pct": pred_upper})
print(ci_df.head(10))
avg_interval_width_pct = ((ci_df["Upper_95pct"] - ci_df["Lower_5pct"]) / ci_df["Predicted"]).mean()
print(f"\nAverage 90% interval width: {avg_interval_width_pct:.1%} of predicted value")


## 11. Serialize the Model (Task 2.5)

In [ ]:

os.makedirs("models", exist_ok=True)
timestamp = datetime.now().strftime("%d-%m-%Y-%H-%M-%S-00")
model_filename = f"models/sales_model_{timestamp}.pkl"
with open(model_filename, "wb") as f:
    pickle.dump({"pipeline": production_pipeline,
                 "model_type": "XGBoost",
                 "feature_cols_num": feature_cols_num,
                 "feature_cols_cat": feature_cols_cat,
                 "median_competition_distance": median_dist}, f)
logger.info(f"Model serialized to {model_filename}")
print("Saved:", model_filename)


## 12. Deep Learning Model — LSTM (Task 2.6)

We build a Long Short-Term Memory (LSTM) network to model sales as a pure time series.
To keep training time practical (the brief specifies the model should comfortably run on limited
compute, e.g. Google Colab), we demonstrate the full pipeline on **one representative, consistently
open store** rather than all 1,115 stores simultaneously — the same methodology scales directly to a
multi-store or store-embedding model given more compute.


In [ ]:

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Pick the store with the most complete, open-every-week history
store_counts = train_clean["Store"].value_counts()
target_store = store_counts.index[0]
print("Selected store for LSTM demo:", target_store)

ts = train_clean[train_clean.Store == target_store].sort_values("Date")[["Date","Sales"]].set_index("Date")
ts = ts.asfreq("D")
ts["Sales"] = ts["Sales"].interpolate()  # fill the rare closed-day gaps left by asfreq
print(ts.shape)
ts.plot(figsize=(14,4), title=f"Daily Sales — Store {target_store}")
plt.ylabel("Sales")
plt.tight_layout()
plt.savefig("plots/lstm_store_timeseries.png", dpi=120)
plt.show()


### 12.1 Isolate the time series & check stationarity

In [ ]:

result = adfuller(ts["Sales"].dropna())
print(f"ADF Statistic: {result[0]:.4f}")
print(f"p-value: {result[1]:.4f}")
if result[1] < 0.05:
    print("=> Series is stationary (reject unit root null hypothesis).")
else:
    print("=> Series is NOT stationary — differencing recommended.")


In [ ]:

ts["Sales_diff"] = ts["Sales"].diff()
result_diff = adfuller(ts["Sales_diff"].dropna())
print(f"After 1st-order differencing — ADF Statistic: {result_diff[0]:.4f}, p-value: {result_diff[1]:.6f}")


### 12.2 Autocorrelation / Partial Autocorrelation

In [ ]:

fig, axes = plt.subplots(1,2, figsize=(14,4))
plot_acf(ts["Sales"].dropna(), lags=30, ax=axes[0])
plot_pacf(ts["Sales"].dropna(), lags=30, ax=axes[1])
axes[0].set_title("ACF - Daily Sales")
axes[1].set_title("PACF - Daily Sales")
plt.tight_layout()
plt.savefig("plots/lstm_acf_pacf.png", dpi=120)
plt.show()


**Insight:** The ADF test on the raw series already rejects the unit-root null (p < 0.05), i.e. the series is stationary in the statistical sense — but it still shows strong weekly (lag-7) autocorrelation in the ACF/PACF plots, confirming a clear 7-day seasonal cycle that the LSTM's lookback window needs to span (we use a 14-day window below to comfortably cover two full weekly cycles).

### 12.3 Supervised learning transform (sliding window) & scaling to (-1, 1)

In [ ]:

from sklearn.preprocessing import MinMaxScaler

series = ts["Sales"].values.reshape(-1,1)
scaler = MinMaxScaler(feature_range=(-1,1))
series_scaled = scaler.fit_transform(series)

WINDOW = 14
def make_supervised(data, window):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i:i+window, 0])
        y.append(data[i+window, 0])
    return np.array(X), np.array(y)

X_seq, y_seq = make_supervised(series_scaled, WINDOW)
X_seq = X_seq.reshape((X_seq.shape[0], X_seq.shape[1], 1))
print("Supervised dataset shape:", X_seq.shape, y_seq.shape)

split = int(len(X_seq)*0.85)
X_seq_train, X_seq_test = X_seq[:split], X_seq[split:]
y_seq_train, y_seq_test = y_seq[:split], y_seq[split:]
print("Train:", X_seq_train.shape, "Test:", X_seq_test.shape)


### 12.4 Build & train a 2-layer LSTM

In [ ]:

import tensorflow as tf
tf.random.set_seed(42)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

lstm_model = Sequential([
    LSTM(32, activation="tanh", return_sequences=True, input_shape=(WINDOW,1)),
    Dropout(0.1),
    LSTM(16, activation="tanh"),
    Dense(1)
])
lstm_model.compile(optimizer="adam", loss="mse")
lstm_model.summary()


In [ ]:

history = lstm_model.fit(X_seq_train, y_seq_train, epochs=20, batch_size=16,
                          validation_split=0.1, verbose=0)
print("Final training loss:", history.history["loss"][-1])
print("Final validation loss:", history.history["val_loss"][-1])

plt.figure(figsize=(8,4))
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.title("LSTM Training Curve")
plt.xlabel("Epoch"); plt.ylabel("MSE (scaled)")
plt.legend()
plt.tight_layout()
plt.savefig("plots/lstm_training_curve.png", dpi=120)
plt.show()


### 12.5 Evaluate on held-out data

In [ ]:

preds_scaled = lstm_model.predict(X_seq_test, verbose=0)
preds_actual = scaler.inverse_transform(preds_scaled)
y_test_actual = scaler.inverse_transform(y_seq_test.reshape(-1,1))

lstm_mae = mean_absolute_error(y_test_actual, preds_actual)
lstm_rmse = np.sqrt(mean_squared_error(y_test_actual, preds_actual))
print(f"LSTM Test MAE: {lstm_mae:,.1f}")
print(f"LSTM Test RMSE: {lstm_rmse:,.1f}")

plt.figure(figsize=(12,5))
plt.plot(y_test_actual, label="Actual")
plt.plot(preds_actual, label="LSTM Predicted")
plt.title(f"LSTM Forecast vs Actual — Store {target_store} (Test Window)")
plt.xlabel("Day index (test period)"); plt.ylabel("Sales")
plt.legend()
plt.tight_layout()
plt.savefig("plots/lstm_forecast.png", dpi=120)
plt.show()


**Insight:** The LSTM tracks the overall level and week-to-week rhythm of sales for this store reasonably well from a 14-day lookback window alone (no external features), though — as expected for a single-store, features-free model — it misses some sharp promo/holiday spikes that the Random Forest (which explicitly sees Promo/holiday features) captures better. This is exactly the trade-off the brief highlights: an LSTM at this scope demonstrates the deep-learning approach on the raw time series, while the feature-rich Random Forest remains the stronger tool for full-scale, all-store production forecasting given today's engineered features.

## 13. Key Findings

1. **Promo effectiveness is highly uneven across stores.** Average promo lift ranges from single digits
   to over 140% depending on the store; a flat, everywhere-promo policy leaves clear money on the table.
2. **Promos genuinely grow the pie, not just accelerate existing spend** — both average Customers and
   average Sales-per-Customer rise on promo days.
3. **Sales and Customers correlate strongly (r ≈ 0.82)**, but Sales-per-Customer varies enough by day/promo
   that Customers alone is not a sufficient forecasting feature.
4. **Strong weekly and yearly seasonality**: Mondays and (where applicable) weekends lead the week; December
   is the clear annual peak, consistent with Christmas trading, with a smaller Easter lift.
5. **Holiday timing matters more than the holiday itself**: sales spike the day *before* a holiday and
   normalize quickly afterward.
6. **Competitor distance is a weak, confounded signal** on its own (r ≈ -0.04) — proximity to competitors
   correlates with slightly higher, not lower, sales, most likely because competitors cluster in strong
   locations.
7. **New competitor openings do not, on average, meaningfully hurt sales** in the sampled stores (mean
   change ≈ -2%), though the range is wide, meaning some stores are more exposed than others.
8. **Assortment 'extra' (b) stores over-index on sales**, though the group is small — worth a closer,
   store-level look before generalizing.
9. **Model comparison** shows a clear accuracy ladder: Linear Regression is weakest (RMSPE ≈ 46%, R² ≈ 0.26),
   Random Forest is a large step up (RMSPE ≈ 28.5%, R² ≈ 0.65), and **XGBoost is a further, substantial
   improvement** (RMSPE ≈ 16.3%, R² ≈ 0.88) on the same features and hold-out — not a marginal difference.
   XGBoost was selected as the production model on this evidence.
10. The **LSTM** demonstrates that sales are learnable from the raw time signal alone (no external
    features), but under-reacts to sharp promo/holiday spikes compared to the feature-aware XGBoost model.


## 14. Business / Project Insights

- **Targeted promotions.** Reallocating promo spend toward the top-lift stores (and away from the
  bottom-lift ones) should raise the average return on promotional spend without increasing total
  promo days run.
- **Forecast-driven inventory planning.** A 6-week-ahead XGBoost forecast, refreshed daily, gives
  finance a materially better planning input than manager intuition, especially heading into
  December/Easter demand surges.
- **Sunday-opening opportunity.** The small set of stores that open on Sundays show sales comparable to
  weekday levels — worth a controlled pilot to open more stores on Sundays where legally permitted.
- **Don't over-weight competitor distance.** Store-siting or defensive-pricing decisions should not be
  based on competitor distance alone; it is a weak and confounded predictor in this data.
- **Confidence-aware forecasting.** The ~86% average width of the 90% prediction interval (from the
  quantile-regression models) is fairly wide with today's default hyperparameters — it should be treated
  as a first pass, tightened via quantile-model tuning, but the *practice* of communicating a range rather
  than a single number is already worth adopting for staffing and stock-ordering decisions.


## 15. Recommendations

1. Move from uniform promo scheduling to a **store-level promo targeting model** using the computed
   per-store lift as a first pass, refined over time with a proper uplift-modelling approach.
2. Put the XGBoost forecasting pipeline into a **daily batch job** (see the Streamlit app built
   below for the interactive front-end) so finance always has a live, 6-week-ahead forecast.
3. Keep comparing new candidate algorithms (e.g. LightGBM, CatBoost) against the same RMSPE/MAE/R²
   benchmark used here before ever replacing the production model, so upgrades stay evidence-based —
   this project's own comparison shows how much that discipline can be worth (a ~12pp RMSPE gap between
   Random Forest and XGBoost alone).
4. Pilot **Sunday openings** at a handful of currently-closed-on-Sunday stores with strong weekday
   performance, and measure the incremental sales lift directly.
5. Extend the current single-store LSTM prototype to a **store-embedding, multi-store LSTM/Temporal model**
   once more compute is available, so the deep-learning approach can also benefit from promo/holiday
   features rather than the raw series alone.


## 16. Final Conclusion

This project delivered an end-to-end sales forecasting pipeline for Rossmann Pharmaceuticals: from
raw, uncleaned multi-store data through a thorough exploratory analysis of customer purchasing behaviour,
into a production-style feature engineering and modelling pipeline that compares three algorithms
(Linear Regression, Random Forest, XGBoost) on identical features and a genuine 6-week forward hold-out,
plus a Deep Learning LSTM prototype demonstrating a features-free time-series alternative. **XGBoost**,
serialized with a timestamp for daily retraining, is the recommended production model given its clearly
superior accuracy (RMSPE ≈ 16.3%, R² ≈ 0.88) over both simpler baselines, backed by feature importances and
a genuine quantile-regression prediction interval — while the model comparison keeps that choice
evidence-based rather than assumed, and the LSTM validates that the underlying sales signal is learnable
from time alone, a natural next investment once multi-store, feature-rich deep learning becomes worthwhile.
A working Streamlit front-end (see `app.py`) lets finance-team analysts request a forecast interactively
and download the results, completing the "serve predictions" requirement of the brief.
